# D_Product

In [ ]:
""" Librerías """

import time
from itertools import combinations

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

import dvgate as dg
import product_loader as PL

SEED, PATH, PILOT, T_SE, N_EXACT_AUD, N_PERM = 99, "datasets/D_Product datos/product_data", 45, 1000, 10, 100_000
T0 = time.time()

## 1 — Carga

In [ ]:
D = PL.load(PATH, n_players=16, min_enc=100, n_strata=4, seed=SEED, verbose=True)
X, y, src, Pl = D["X"], D["y"], D["src"], D["players"]
TR, VA, TE = D["idx_tr"], D["idx_va"], D["idx_te"]

v, v0, idxp = dg.make_game(X, y, src, TR, VA, Pl, cap=None, seed=SEED)
print(f"\nv(vacío)={v0:.4f}  v(N)={v(Pl):.4f}  excedente={v(Pl) - v0:.4f}")

## 2 — Panel

In [ ]:
pan = dg.run_panel(X, y, src, TR, VA, TE, Pl, v, D["meta"], SEED, PILOT, T_SE)
print(f"\n{pan['tabla'].to_string(index=False)}")
print(f"\nconclusion: {pan['veredicto']} — {pan['motivo']}")

## 3 — Shapley exacto

In [ ]:
t = time.time()
yva = y[VA]
SC = {}
for k in range(len(Pl) + 1):
    for S in combinations(Pl, k):
        if not S:
            continue
        r = np.concatenate([idxp[p] for p in S])
        if len(r) < 30 or len(np.unique(y[r])) < 2:
            continue
        SC[frozenset(S)] = dg._fit(X[r], y[r], SEED).predict_proba(X[VA])[:, 1].astype(np.float32)

def v_exact(S):
    S = [p for p in S if p in idxp]
    if not S:
        return v0
    sc = SC.get(frozenset(S))
    return v0 if sc is None else float(average_precision_score(yva, sc))

phi = dg.exact_shapley(Pl, v_exact)
min_caro = (time.time() - t) / 60
efic = sum(phi.values()) - (v_exact(Pl) - v0)
print(f"\nShapley exacto: {len(SC):,} coaliciones cacheadas | eficiencia "
      f"suma-(v(N)-v0)={efic:+.2e} | {min_caro:.2f} min")

_, se, _, _ = dg.tmc_shapley(Pl, v, T_SE, SEED)
lo = dg.loo(Pl, v)
cl, guiada = dg.close_loop(X, y, src, TR, TE, Pl, phi, se, D["meta"], SEED)
print(f"\nCARO (fase de decisión, SE de TMC T={T_SE}): phi<0 en "
      f"{sorted(k for k in Pl if phi[k] < 0)} | regla phi+2SE<0 marca {guiada}")
print(cl[["k", "contexto", "auroc", "ap", "d_ap", "d_ap_lo", "d_ap_hi"]].round(4).to_string())

## 4 — ahapley exacto y tmc_shapley , pero sobre 10 jugadores

In [ ]:
SUB = sorted(np.random.default_rng(SEED + 1).choice(Pl, N_EXACT_AUD, replace=False).tolist())
v_sub, _, _ = dg.make_game(X, y, src, TR, VA, SUB, cap=None, seed=SEED)
ex_sub = dg.exact_shapley(SUB, v_sub)
tm_sub, _, _, _ = dg.tmc_shapley(SUB, v_sub, T_SE, SEED)
rho_c = float(pd.Series(ex_sub).rank().corr(pd.Series(tm_sub).rank()))
print(f"rho(exacto, TMC) en subconjunto de {N_EXACT_AUD}={rho_c:.3f} | "
      f"comparamos rho={pan['resolucion']['rho_esperado']:.3f}")

## 5 — correlacionamos SPEARMAN, vemos significancia -> con test permutaciones & asintotica

In [ ]:
AUD = D["calidad_real"].copy()
AUD["error_real"] = 1 - AUD.precision_real
AUD["phi"] = pd.Series(phi)
AUD["marcada_panel"] = pan["daño"]["dañina"]
AUD["ret"] = pan["daño"]["ret"]
AUD["tipo"] = pan["daño"]["tipo"]
AUD["loo"] = pd.Series(lo)
print(AUD.sort_values("precision_real").round(4).to_string())

r_e = AUD.error_real.rank().values
r_p = AUD.phi.rank().values
n = len(AUD)
rho_phi = float(np.corrcoef(r_p, r_e)[0, 1])
t_stat = rho_phi * np.sqrt((n - 2) / (1 - rho_phi ** 2))
try:
    from scipy import stats
    p_t = 2 * stats.t.sf(abs(t_stat), df=n - 2)
except ImportError:
    p_t = None

rng = np.random.default_rng(SEED)
perm_rhos = np.empty(N_PERM)
for i in range(N_PERM):
    perm_rhos[i] = np.corrcoef(r_p, rng.permutation(r_e))[0, 1]
p_perm = float((np.abs(perm_rhos) >= abs(rho_phi) - 1e-12).mean())

print(f"\nrho(rango phi_exacto, error_real) = {rho_phi:+.4f}  (n={n}, se espera negativo)\n"
      f"  aprox. t: t={t_stat:+.3f}, df={n - 2}, "
      f"p={'%.4f' % p_t if p_t is not None else 'scipy no disponible'} (bilateral)\n"
      f"  test de permutaciones ({N_PERM:,} réplicas): p={p_perm:.4f} (bilateral)")

## 6 — Cierre

In [ ]:
print(f"\ntotal {(time.time() - T0) / 60:.2f} min | conclusiones: {pan['veredicto']}")
VAL = pd.DataFrame({"phi": pd.Series(phi), "se_TMC_T1000": pd.Series(se), "loo": pd.Series(lo)})
for nom, obj in [("product_valores", VAL), ("product_dano", pan["daño"]),
                 ("product_auditoria", AUD), ("product_cierre_bucle", cl)]:
    obj.to_csv(f"{nom}.csv")
print("ficheros on resultados: product_valores.csv product_dano.csv product_auditoria.csv "
      "product_cierre_bucle.csv")